In [ ]:
#!/usr/bin/env python
# coding: utf-8

import docker
import pandas as pd
import time

def get_container_stats(container, data_list):
    """
    Obtient les statistiques du conteneur et les ajoute à la liste de données.
    """
    stats_stream = container.stats(stream=True, decode=True)
    stats = next(stats_stream)
    
    cpu_stats = stats['cpu_stats']
    cpu_percentage = calculate_cpu_percentage(cpu_stats)
    
    mem_usage = stats['memory_stats']['usage'] / (1024 * 1024)
    mem_limit = stats['memory_stats']['limit'] / (1024 * 1024)
    mem_percentage = (mem_usage / mem_limit) * 100.0
    
    net_io = f"{stats['networks']['eth0']['rx_bytes'] / 1024:.2f}KB / {stats['networks']['eth0']['tx_bytes'] / 1024:.2f}KB"
    block_io = f"{stats['blkio_stats']['io_service_bytes_recursive'][0]['value'] / (1024 * 1024):.2f}MB / {stats['blkio_stats']['io_service_bytes_recursive'][1]['value'] / (1024 * 1024):.2f}MB"
    pids = stats['pids_stats']['current']
    
    current_time = time.strftime("%Y-%m-%d %H:%M:%S")
    
    data_list.append([current_time, cpu_percentage, mem_usage, mem_limit, mem_percentage, net_io, block_io, pids])

def calculate_cpu_percentage(cpu_stats):
    """
    Calcule le pourcentage d'utilisation du CPU.
    """
    cpu_percentage = 0.0
    if 'cpu_usage' in cpu_stats and 'total_usage' in cpu_stats['cpu_usage']:
        cpu_percentage = (cpu_stats['cpu_usage']['total_usage'] / cpu_stats['system_cpu_usage']) * 100.0
    return cpu_percentage

def main():
    container_name = "Nginx-Unit-Python"
    client = docker.from_env()

    try:
        container = client.containers.get(container_name)
    except docker.errors.NotFound:
        print(f"Le conteneur {container_name} n'a pas été trouvé.")
        return

    data_list = []
    collection_duration = 600
    collection_interval = 1
    start_time = time.time()

    while time.time() - start_time < collection_duration:
        print(time.time() - start_time)
        get_container_stats(container, data_list)
        time.sleep(collection_interval)

    df = pd.DataFrame(data_list, columns=["Time", "CPU %", "MEM USAGE (MB)", "MEM LIMIT (MB)", "MEM %", "NET I/O", "BLOCK I/O", "PIDS"])
    df.to_csv("container_stats_unit.csv", sep=";", index=False)
    print("Données enregistrées dans container_stats_unit.csv")

if __name__ == "__main__":
    main()
